In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset

# 101 row QA
ds1 = load_dataset("prsdm/Machine-Learning-QA-dataset")
# 64 row QA
ds2 = load_dataset("whiteOUO/Ladder-machine-learning-QA")
# 473row qa
ds3 = load_dataset("team-bay/data-science-qa")
# 508 qa
ds4 = load_dataset("mjphayes/machine_learning_questions")
# 1.13k qa
ds5 = load_dataset("Harikrishnan46624/AI_QA_Data")
# 1.07k QA
ds6 = load_dataset("soufyane/DATA_SCIENCE_QA")
# 6.22k QA
ds7 = load_dataset("RazinAleks/SO-Python_QA-Data_Science_and_Machine_Learning_class")

Repo card metadata block was not found. Setting CardData to empty.


In [2]:
# convert hugging face datasets into pandas DataFrame
def convert(dataset):
    return pd.DataFrame(dataset)

In [3]:
df4_1 = convert(ds4["train"])
df4_2 = convert(ds4["test"])
df4 = pd.concat([df4_1,df4_2])
df4 = df4[['question','answer']]

In [4]:
df7_0 = convert(ds7["train"])
df7_1 = convert(ds7["validation"])
df7_2 = convert(ds7["test"])
df7 = pd.concat([df7_0,df7_1,df7_2])
df7 = df7[['Question','Answer']]

In [5]:
df1, df2, df3, df5, df6 = map(convert,(ds1['train'], ds2['train'], ds3['train'], ds5['train'], ds6['train']))

df1 = df1[['Question','Answer']]
df2 = df2[['Question','Answer']]
df3 = df3[['question','answer']]
df5 = df5[['question','answer']]
df6 = df6[['Question','Answer']]

In [6]:
df3.rename(columns={'question':'Question','answer':'Answer'},inplace=True)
df4.rename(columns={'question':'Question','answer':'Answer'},inplace=True)
df5.rename(columns={'question':'Question','answer':'Answer'},inplace=True)

df = pd.concat([df1,df2,df3,df4,df5,df6,df7])

In [7]:
def formatting(row: pd.Series) -> str:
    '''This function meant to format data entry into the way that gemma required
    data. Note, it doesn't contain <bos> or <eos> 
    '''
    text = '''<start_of_turn>user
    {}<end_of_turn>
    <start_of_turn>model
    {}<end_of_turn>'''.format(row["Question"],row["Answer"])
    return text

In [8]:
processed_data = df.apply(formatting, axis=1)  

In [9]:
# split all data into train, dev and test sets

np.random.seed(66)
perm = np.random.permutation(len(processed_data))
dev_size = int(0.1 * len(processed_data))
test_size = int(0.2 * len(processed_data))

train_set = [processed_data.iloc[i] for i in perm[test_size + dev_size:]]
dev_set = [processed_data.iloc[i] for i in perm[test_size:test_size + dev_size]]
test_set = [processed_data.iloc[i] for i in perm[:test_size]]

In [10]:
# Save all datasets

pd.DataFrame(train_set,columns={'text'}).to_json("data/train.jsonl", orient="records", lines=True, force_ascii=False)
pd.DataFrame(dev_set,columns={'text'}).to_json("data/valid.jsonl", orient="records", lines=True, force_ascii=False)
pd.DataFrame(test_set,columns={'text'}).to_json("data/test.jsonl", orient="records", lines=True, force_ascii=False)